In [1]:
# ============================================
# IPL Win Probability Predictor
# Day 2 — Data Exploration
# Phase 3 — Understanding Match Structure
# ============================================
# GOAL: Deeply understand one match, calculate 
# ball-by-ball score, wickets, and run rates

import pandas as pd
import numpy as np

matches = pd.read_csv("../data/matches.csv")
deliveries = pd.read_csv("../data/deliveries.csv")

print("✅ Data reloaded — Shape:", matches.shape, deliveries.shape)

✅ Data reloaded — Shape: (756, 18) (179078, 21)


In [2]:
sample_match_id=matches["id"].iloc[0]
print("Studying Match id :",sample_match_id)

# Get all deliveries from this match
sample_match=deliveries[deliveries["match_id"]==sample_match_id]

print("\n Total balls bowled in the match",len(sample_match))
print("\n Innings present:", sample_match["inning"].unique())

sample_match

Studying Match id : 1

 Total balls bowled in the match 248

 Innings present: [1 2]


,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,4,0,4,NaN,NaN,NaN
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,2,2,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,19,6,YS Chahal,A Choudhary,B Kumar,0,...,0,0,0,0,1,0,1,NaN,NaN,NaN
244,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,20,1,YS Chahal,A Choudhary,BCJ Cutting,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
245,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,20,2,YS Chahal,A Choudhary,BCJ Cutting,0,...,0,0,0,0,1,0,1,NaN,NaN,NaN
246,1,2,Royal Challengers Bangalore,Sunrisers Hyderabad,20,3,A Choudhary,YS Chahal,BCJ Cutting,0,...,0,0,0,0,6,0,6,NaN,NaN,NaN


In [3]:
# Separate innings 1 and innings 2
inning1=sample_match[sample_match["inning"]==1]
inning2=sample_match[sample_match["inning"]==2]

print("Inning 1 - balls bowled : ",len(inning1))
print("Total runs : ",inning1["total_runs"].sum())

print("Inning 2 - balls bowled : ",len(inning2))
print("Total runs : ",inning2["total_runs"].sum())

Inning 1 - balls bowled :  125
Total runs :  207
Inning 2 - balls bowled :  123
Total runs :  172


In [4]:
# The target for innings 2 = innings 1 total runs + 1
target=inning1["total_runs"].sum()+1
print(f"Team batting second needs {target} runs to win")

# Confirm with matches.csv
match_info=matches[matches["id"]==sample_match_id]
print(match_info[['team1', 'team2', 'winner', 'win_by_runs', 'win_by_wickets']])

Team batting second needs 208 runs to win
                 team1                        team2               winner  \
0  Sunrisers Hyderabad  Royal Challengers Bangalore  Sunrisers Hyderabad   

   win_by_runs  win_by_wickets  
0           35               0  


In [5]:
# Make a copy to avoid modifying original data
inning2=inning2.copy()

# Cumulative score after each ball
inning2['current_score']=inning2['total_runs'].cumsum()

# Cumulative wickets after each ball
# player_dismissed is NaN when no wicket falls, so we check if it's NOT NaN
inning2['is_wicket']=inning2['player_dismissed'].notna().astype(int)
inning2['wicket_fallen']=inning2['is_wicket'].cumsum()

# Show progress
inning2[['over', 'ball', 'batsman_runs', 'total_runs', 
          'current_score', 'is_wicket', 'wicket_fallen']].head(20)

,over,ball,batsman_runs,total_runs,current_score,is_wicket,wicket_fallen
125,1,1,1,1,1,0,0
126,1,2,0,0,1,0,0
127,1,3,0,0,1,0,0
128,1,4,2,2,3,0,0
129,1,5,4,4,7,0,0
130,1,6,4,4,11,0,0
131,2,1,0,0,11,0,0
132,2,2,0,0,11,0,0
133,2,3,1,1,12,0,0
134,2,4,0,0,12,0,0


In [6]:
# Runs remaining to reach target
inning2['runs_remaining']=target-inning2['current_score']

# Balls bowled so far (over * 6 + ball, since 6 balls per over)
inning2['balls_bowled']=(inning2['over']-1)*6 + inning2['ball']

# Balls remaining (20 overs = 120 balls total in T20)
inning2['balls_remaining']=120-inning2['balls_bowled']

# Current Run Rate (CRR) = runs scored per over so far
inning2['current_run_rate']=(inning2['current_score']*6)/inning2['balls_bowled']

# Required Run Rate (RRR) = runs needed per over remaining
inning2['required_run_rate']=(inning2['runs_remaining']*6)/inning2['balls_remaining']

inning2[['over', 'ball', 'current_score', 'wicket_fallen', 
          'runs_remaining', 'balls_remaining', 
          'current_run_rate', 'required_run_rate']].tail(15)



,over,ball,current_score,wicket_fallen,runs_remaining,balls_remaining,current_run_rate,required_run_rate
233,18,2,156,6,52,16,9.000000,19.500000
234,18,3,156,7,52,15,8.914286,20.800000
235,18,4,156,8,52,14,8.830189,22.285714
236,18,5,156,8,52,13,8.747664,24.000000
237,18,6,156,8,52,12,8.666667,26.000000
238,19,1,157,8,51,11,8.642202,27.818182
239,19,2,157,8,51,10,8.563636,30.600000
240,19,3,158,8,50,9,8.540541,33.333333
241,19,4,164,8,44,8,8.785714,33.000000
242,19,5,164,9,44,7,8.707965,37.714286


In [7]:
# Demonstrating the problem directly
runs_remaining_example = 10
balls_remaining_example = 0

try:
    result = (runs_remaining_example * 6) / balls_remaining_example
    print("Result:", result)
except ZeroDivisionError as e:
    print("Error caught:", e)

Error caught: division by zero
